In [602]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import os

In [603]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.gru(x); out = self.dropout(out[:, -1, :]); out = self.fc(out)
        return out

In [604]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_prob):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_prob if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.lstm(x); out = self.dropout(out[:, -1, :]); out = self.fc(out)
        return out

In [605]:
def create_sequences(data, n_steps, target_col):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data.iloc[i:(i + n_steps)].values)
        y.append(data.iloc[i + n_steps][target_col])
    return np.array(X), np.array(y).reshape(-1, 1)

In [606]:
def calculate_mape(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    nonzero_mask = actual != 0
    if not np.any(nonzero_mask): return float('inf')
    return np.mean(np.abs((actual[nonzero_mask] - predicted[nonzero_mask]) / actual[nonzero_mask])) * 100

In [607]:
def get_predictions(model, loader, device):
    model.eval()
    all_predictions = []
    with torch.no_grad():
        for batch_X, _ in loader:
            outputs = model(batch_X.to(device))
            all_predictions.append(outputs.cpu().numpy())
    return np.concatenate(all_predictions)

In [608]:
def inverse_transform_predictions(predictions, original_df, target_col_name, scaler_obj):
    price_col_index = original_df.columns.get_loc(target_col_name)
    dummy_array = np.zeros((len(predictions), original_df.shape[1]))
    dummy_array[:, price_col_index] = predictions.flatten()
    return scaler_obj.inverse_transform(dummy_array)[:, price_col_index]

In [609]:
CROP_NAME = 'Turmeric'
CROP_CSV_PATH = f'../5_Weather_Merged_CSVs/{CROP_NAME}.csv'

In [ ]:
GRU_MODEL_PATH = f'../7_Results/GRU_Results/{CROP_NAME}_best_model_gru.pth'
GRU_OPTIMIZED_MODEL_PATH = f'../7_Results/GRU_Results_Optimized/{CROP_NAME}_best_model_gru_optimized.pth'
LSTM_MODEL_PATH = f'../7_Results/LSTM_Results/{CROP_NAME}_best_model_lstm.pth'
LSTM_OPTIMIZED_MODEL_PATH = f'../7_Results/LSTM_Results_Optimized/{CROP_NAME}_best_model_lstm_optimized.pth'

In [611]:
BEST_N_STEPS = 14
print(f"--- Starting Ensemble Analysis for: {CROP_NAME} with n_steps = {BEST_N_STEPS} ---")

--- Starting Ensemble Analysis for: Turmeric with n_steps = 14 ---


In [612]:
df = pd.read_csv(CROP_CSV_PATH)
df['Price Date'] = pd.to_datetime(df['Price Date'])
df.set_index('Price Date', inplace=True); df.sort_index(inplace=True)

# Feature Engineering
df['day_of_year'] = df.index.dayofyear; df['week_of_year'] = df.index.isocalendar().week.astype(int); df['month'] = df.index.month
categorical_cols = ['District Name', 'Market Name', 'Commodity', 'Variety', 'Grade']
for col in categorical_cols:
    if df[col].dtype == 'object': df[col] = LabelEncoder().fit_transform(df[col])

In [613]:
scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
target_column = 'Modal Price (Rs./Quintal)'

X, y = create_sequences(df_scaled, BEST_N_STEPS, target_column)

train_val_split = int(0.8 * len(X))
X_train_val, X_test = X[:train_val_split], X[train_val_split:]
y_train_val, y_test = y[:train_val_split], y[train_val_split:]
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float()
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)
print(f"Test data created with {len(X_test)} samples.")

Test data created with 3222 samples.


In [614]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = X_test.shape[2]

In [615]:
model_params = {'input_size': input_size, 'hidden_size': 50, 'num_layers': 2, 'output_size': 1, 'dropout_prob': 0.2}

print("\nLoading trained models...")
model_gru_gen = GRUModel(**model_params).to(device)
model_gru_gen.load_state_dict(torch.load(GRU_MODEL_PATH, map_location=device))

model_gru_opt = GRUModel(**model_params).to(device)
model_gru_opt.load_state_dict(torch.load(GRU_OPTIMIZED_MODEL_PATH, map_location=device))

model_lstm_gen = LSTMModel(**model_params).to(device)
model_lstm_gen.load_state_dict(torch.load(LSTM_MODEL_PATH, map_location=device))

model_lstm_opt = LSTMModel(**model_params).to(device)
model_lstm_opt.load_state_dict(torch.load(LSTM_OPTIMIZED_MODEL_PATH, map_location=device))
print("All models loaded successfully.")


Loading trained models...
All models loaded successfully.


In [616]:
print("\nGenerating predictions from each model...")
pred_gru_gen_scaled = get_predictions(model_gru_gen, test_loader, device)
pred_gru_opt_scaled = get_predictions(model_gru_opt, test_loader, device)
pred_lstm_gen_scaled = get_predictions(model_lstm_gen, test_loader, device)
pred_lstm_opt_scaled = get_predictions(model_lstm_opt, test_loader, device)

# Inverse transform to get actual prices
actual_prices = inverse_transform_predictions(y_test, df_scaled, target_column, scaler)
pred_gru_gen = inverse_transform_predictions(pred_gru_gen_scaled, df_scaled, target_column, scaler)
pred_gru_opt = inverse_transform_predictions(pred_gru_opt_scaled, df_scaled, target_column, scaler)
pred_lstm_gen = inverse_transform_predictions(pred_lstm_gen_scaled, df_scaled, target_column, scaler)
pred_lstm_opt = inverse_transform_predictions(pred_lstm_opt_scaled, df_scaled, target_column, scaler)
print("Predictions generated and inverse transformed.")


Generating predictions from each model...
Predictions generated and inverse transformed.


In [617]:
print("\nCreating and evaluating the ensemble...")
# Simple averaging ensemble
ensemble_predictions = (pred_gru_gen + pred_gru_opt + pred_lstm_gen + pred_lstm_opt) / 4


Creating and evaluating the ensemble...


In [618]:
models_to_evaluate = {
    "GRU General": pred_gru_gen,
    "GRU Optimized": pred_gru_opt,
    "LSTM General": pred_lstm_gen,
    "LSTM Optimized": pred_lstm_opt,
    "ENSEMBLE (Average)": ensemble_predictions
}

evaluation_results = []
for name, predictions in models_to_evaluate.items():
    rmse = np.sqrt(mean_squared_error(actual_prices, predictions))
    mape = calculate_mape(actual_prices, predictions)
    evaluation_results.append({
        "Model": name,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

results_summary_df = pd.DataFrame(evaluation_results).sort_values(by="MAPE (%)")

print("\n--- Final Performance Comparison ---")
print(results_summary_df.to_string(index=False))


--- Final Performance Comparison ---
             Model        RMSE  MAPE (%)
       GRU General 1300.575321  9.714185
     GRU Optimized 1691.838588 11.910057
ENSEMBLE (Average) 1453.531135 12.324401
      LSTM General 1881.613273 18.032237
    LSTM Optimized 1852.688291 21.272244


In [ ]:
final_predictions_df = pd.DataFrame({
    'Actual_Price': actual_prices,
    'GRU_General_Pred': pred_gru_gen,
    'GRU_Optimized_Pred': pred_gru_opt,
    'LSTM_General_Pred': pred_lstm_gen,
    'LSTM_Optimized_Pred': pred_lstm_opt,
    'Ensemble_Pred': ensemble_predictions
})

output_filename = f"../7_Results/Ensemble_Results/{CROP_NAME}_Ensemble_Predictions.csv"
final_predictions_df.to_csv(output_filename, index=False)
print(f"\nDetailed predictions saved to '{output_filename}'")


Detailed predictions saved to './Ensemble_Results/Turmeric_Ensemble_Predictions.csv'
